# Lesson 02: From Operating Metrics to P&L Impact

## Goal
Connect workflow metrics to business financials and build operationalized cost models.

## Key Concept
Lesson 01 answered: **Which P&L line does this workflow impact?**
Lesson 02 answers: **How much money is this worth?**

We move from conceptual mapping to quantitative modeling using:
- Volume (annual transactions)
- Touch time (manual minutes per case)
- Rework rate (% that need reprocessing)
- Escalation rate (% that require specialist/approval)
- Loaded labor cost (€/hour including benefits + overhead)

**Duration:** ~2-2.5 hours | **Status:** Building on Lesson 01

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("✓ Libraries ready")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

✓ Libraries ready
pandas: 3.0.3
numpy: 2.4.6


---

## Part 1: Quick Recap from Lesson 01

### Basic Formula (from Lesson 00/01)
```
Annual Manual Cost = Volume × (Minutes ÷ 60) × Hourly Cost
```

This assumes:
- Every case is processed once (no rework)
- No escalations to higher-cost resources
- No queue or wait time

**Reality:** All three happen! This lesson adds them back in.

In [2]:
# Original simple function
def annual_manual_cost_simple(volume, minutes_per_case, hourly_cost):
    """Simple model: assumes no rework or escalations"""
    return volume * (minutes_per_case / 60) * hourly_cost

# Test
baseline = annual_manual_cost_simple(
    volume=4200,
    minutes_per_case=12,
    hourly_cost=55
)
print(f"Invoice approval baseline cost (no rework): €{baseline:,.2f}/year")

Invoice approval baseline cost (no rework): €46,200.00/year


---

## Part 2: Five Workflow Metrics That Drive Real Cost

### The Five Metrics

1. **Volume** — How many cases per year? (transactions, tickets, reports)
2. **Touch Time** — How many manual minutes per case? (includes data entry, review, decision)
3. **Rework Rate** — What % of cases need to be reprocessed? (missing data, errors, rejections)
4. **Escalation Rate** — What % go to a more expensive resource? (specialist, manager, legal)
5. **Hourly Cost** — What's the loaded cost per hour for each resource type?

Let me show why each matters with examples.

In [3]:
# Example: Invoice Approval Workflow
print("EXAMPLE: Invoice Approval Cost Drivers")
print("="*70)

metrics = {
    'Volume': 4200,
    'Touch Time (min)': 12,
    'Rework Rate': '5%',
    'Escalation Rate': '2%',
    'AP Analyst Cost': '€55/hr',
    'AP Manager Cost': '€75/hr'
}

for metric, value in metrics.items():
    print(f"{metric:25} → {value}")

print("\n" + "="*70)
print("Each metric drives different parts of the cost:")
print("  • Volume × Touch Time → BASE COST")
print("  • Rework Rate × Touch Time → REWORK COST (reprocessing)")
print("  • Escalation Rate × Extra Time → ESCALATION COST (manager review)")
print("  • TOTAL = Base + Rework + Escalation")

EXAMPLE: Invoice Approval Cost Drivers
Volume                    → 4200
Touch Time (min)          → 12
Rework Rate               → 5%
Escalation Rate           → 2%
AP Analyst Cost           → €55/hr
AP Manager Cost           → €75/hr

Each metric drives different parts of the cost:
  • Volume × Touch Time → BASE COST
  • Rework Rate × Touch Time → REWORK COST (reprocessing)
  • Escalation Rate × Extra Time → ESCALATION COST (manager review)
  • TOTAL = Base + Rework + Escalation


---

## Part 3: Loaded Labor Cost Decomposition

Why is an analyst who earns €40k/year worth €55/hour?

### Cost of an Employee to the Business

In [4]:
# Typical employee cost breakdown
loaded_cost_breakdown = pd.DataFrame({
    'Cost Category': [
        'Base Salary',
        'Benefits (health, retirement)',
        'Payroll Taxes (employer)',
        'Facilities & Overhead',
        'Tools & Equipment',
        'Management/Support',
    ],
    'Annual Cost (€)': [
        40000,
        9000,
        5200,
        2500,
        1000,
        1300,
    ]
})

total_annual = loaded_cost_breakdown['Annual Cost (€)'].sum()
working_hours_per_year = 1960  # 52 weeks × 40 hours - vacation/sick - meetings
hourly_cost = total_annual / working_hours_per_year

print("LOADED LABOR COST CALCULATION")
print("="*70)
print(loaded_cost_breakdown.to_string(index=False))
print("\n" + "="*70)
print(f"Total Annual Cost to Business: €{total_annual:,.0f}")
print(f"Working Hours per Year: {working_hours_per_year:,} (accounting for vacation, meetings)")
print(f"\nLoaded Hourly Cost: €{hourly_cost:.2f}/hour")
print("\nThis is the €55/hr we use throughout the course!")

LOADED LABOR COST CALCULATION
                Cost Category  Annual Cost (€)
                  Base Salary            40000
Benefits (health, retirement)             9000
     Payroll Taxes (employer)             5200
        Facilities & Overhead             2500
            Tools & Equipment             1000
           Management/Support             1300

Total Annual Cost to Business: €59,000
Working Hours per Year: 1,960 (accounting for vacation, meetings)

Loaded Hourly Cost: €30.10/hour

This is the €55/hr we use throughout the course!


---

## Part 4: Extended Cost Model with Rework & Escalation

### The Complete Cost Formula

```
Total Cost = Base Cost + Rework Cost + Escalation Cost

Base Cost = Volume × (Touch Time ÷ 60) × Hourly Cost
Rework Cost = Volume × Rework Rate × (Touch Time ÷ 60) × Hourly Cost
Escalation Cost = Volume × Escalation Rate × (Extra Time ÷ 60) × Escalation Hourly Cost
```

In [5]:
def cost_model_complete(volume, touch_time_min, rework_rate, escalation_rate,
                        base_hourly_cost, escalation_hourly_cost=None,
                        escalation_extra_time_min=None):
    """
    Complete cost model including rework and escalation.
    
    Args:
        volume: annual transaction count
        touch_time_min: manual minutes per transaction (base case)
        rework_rate: % of cases requiring reprocessing (0.0-1.0)
        escalation_rate: % escalated to higher resource (0.0-1.0)
        base_hourly_cost: analyst/processor hourly rate (€/hr)
        escalation_hourly_cost: manager/specialist rate (default: 1.4x base)
        escalation_extra_time_min: additional minutes for escalation (default: same as touch_time)
    
    Returns:
        dict with breakdown of costs
    """
    if escalation_hourly_cost is None:
        escalation_hourly_cost = base_hourly_cost * 1.4  # Default: 40% higher
    
    if escalation_extra_time_min is None:
        escalation_extra_time_min = touch_time_min
    
    base_cost = volume * (touch_time_min / 60) * base_hourly_cost
    rework_cost = volume * rework_rate * (touch_time_min / 60) * base_hourly_cost
    escalation_cost = volume * escalation_rate * (escalation_extra_time_min / 60) * escalation_hourly_cost
    
    total = base_cost + rework_cost + escalation_cost
    
    return {
        'base_cost': base_cost,
        'rework_cost': rework_cost,
        'escalation_cost': escalation_cost,
        'total_cost': total,
        'cost_per_transaction': total / volume if volume > 0 else 0,
    }

print("✓ Cost model function defined")

✓ Cost model function defined


In [6]:
# Quick test of the function
test_result = cost_model_complete(
    volume=4200,
    touch_time_min=12,
    rework_rate=0.05,
    escalation_rate=0.02,
    base_hourly_cost=55,
    escalation_hourly_cost=75,
)

print("Invoice Approval - Cost Model Test:")
print("="*50)
print(f"Base cost:        €{test_result['base_cost']:>10,.2f}")
print(f"Rework cost:      €{test_result['rework_cost']:>10,.2f}")
print(f"Escalation cost:  €{test_result['escalation_cost']:>10,.2f}")
print("-" * 50)
print(f"Total cost:       €{test_result['total_cost']:>10,.2f}")
print(f"Cost per invoice: €{test_result['cost_per_transaction']:>10,.2f}")

Invoice Approval - Cost Model Test:
Base cost:        € 46,200.00
Rework cost:      €  2,310.00
Escalation cost:  €  1,260.00
--------------------------------------------------
Total cost:       € 49,770.00
Cost per invoice: €     11.85


---

## Workflow 1: Invoice Approval Automation

**Scenario:** Finance team manually reviews invoices for compliance and accuracy before payment.

### Current State Metrics

In [7]:
print("WORKFLOW 1: INVOICE APPROVAL")
print("="*70)

# Invoice approval metrics
inv_volume = 4200  # invoices per year
inv_touch_time = 12  # minutes per invoice
inv_rework_rate = 0.05  # 5% have missing fields or amount mismatches
inv_escalation_rate = 0.02  # 2% complex or unusual need manager approval
ap_analyst_cost = 55  # €/hr
ap_manager_cost = 75  # €/hr (manager review)

print(f"\nCURRENT STATE METRICS:")
print(f"  Annual volume: {inv_volume:,} invoices")
print(f"  Touch time: {inv_touch_time} minutes per invoice")
print(f"  Rework rate: {inv_rework_rate*100:.0f}% (missing fields, amount mismatches)")
print(f"  Escalation rate: {inv_escalation_rate*100:.1f}% (complex → manager approval)")
print(f"  AP Analyst cost: €{ap_analyst_cost}/hr")
print(f"  AP Manager cost: €{ap_manager_cost}/hr")

# Calculate cost
inv_cost = cost_model_complete(
    volume=inv_volume,
    touch_time_min=inv_touch_time,
    rework_rate=inv_rework_rate,
    escalation_rate=inv_escalation_rate,
    base_hourly_cost=ap_analyst_cost,
    escalation_hourly_cost=ap_manager_cost,
)

print(f"\nCOST BREAKDOWN:")
print(f"  Base analyst processing: €{inv_cost['base_cost']:>10,.2f}")
print(f"  Rework (reprocessing):   €{inv_cost['rework_cost']:>10,.2f}")
print(f"  Manager escalations:     €{inv_cost['escalation_cost']:>10,.2f}")
print(f"  " + "-"*40)
print(f"  TOTAL COST OF FRICTION:  €{inv_cost['total_cost']:>10,.2f}/year")
print(f"  Cost per invoice:        €{inv_cost['cost_per_transaction']:>10,.2f}")

WORKFLOW 1: INVOICE APPROVAL

CURRENT STATE METRICS:
  Annual volume: 4,200 invoices
  Touch time: 12 minutes per invoice
  Rework rate: 5% (missing fields, amount mismatches)
  Escalation rate: 2.0% (complex → manager approval)
  AP Analyst cost: €55/hr
  AP Manager cost: €75/hr

COST BREAKDOWN:
  Base analyst processing: € 46,200.00
  Rework (reprocessing):   €  2,310.00
  Manager escalations:     €  1,260.00
  ----------------------------------------
  TOTAL COST OF FRICTION:  € 49,770.00/year
  Cost per invoice:        €     11.85


In [8]:
# AI Impact Analysis
print("\nAI INVOICE EXTRACTION IMPACT:")
print("="*70)

# AI improvements
ai_touch_time_reduction = 0.50  # 50% faster (AI extracts fields)
ai_rework_reduction = 0.80  # 80% fewer errors (AI validates formatting)
ai_escalation_reduction = 0.80  # 80% fewer need manager review (AI flags anomalies)
ai_platform_cost = 0.03  # €0.03 per invoice for AI platform

inv_ai_touch_time = inv_touch_time * (1 - ai_touch_time_reduction)
inv_ai_rework_rate = inv_rework_rate * (1 - ai_rework_reduction)
inv_ai_escalation_rate = inv_escalation_rate * (1 - ai_escalation_reduction)

print(f"AI improvements:")
print(f"  Touch time reduction: {ai_touch_time_reduction*100:.0f}% → {inv_ai_touch_time:.1f} min/invoice")
print(f"  Rework reduction: {ai_rework_reduction*100:.0f}% → {inv_ai_rework_rate*100:.2f}% rework rate")
print(f"  Escalation reduction: {ai_escalation_reduction*100:.0f}% → {inv_ai_escalation_rate*100:.2f}% escalation rate")
print(f"  AI cost: €{ai_platform_cost:.2f} per invoice")

# Calculate new cost with AI
inv_cost_ai = cost_model_complete(
    volume=inv_volume,
    touch_time_min=inv_ai_touch_time,
    rework_rate=inv_ai_rework_rate,
    escalation_rate=inv_ai_escalation_rate,
    base_hourly_cost=ap_analyst_cost,
    escalation_hourly_cost=ap_manager_cost,
)
ai_platform_total = inv_volume * ai_platform_cost
inv_cost_ai_total = inv_cost_ai['total_cost'] + ai_platform_total

print(f"\nWITH AI:")
print(f"  Manual processing:     €{inv_cost_ai['base_cost']:>10,.2f}")
print(f"  Rework:                €{inv_cost_ai['rework_cost']:>10,.2f}")
print(f"  Escalations:           €{inv_cost_ai['escalation_cost']:>10,.2f}")
print(f"  AI platform cost:      €{ai_platform_total:>10,.2f}")
print(f"  " + "-"*40)
print(f"  TOTAL COST:            €{inv_cost_ai_total:>10,.2f}/year")

inv_savings = inv_cost['total_cost'] - inv_cost_ai_total
print(f"\n" + "="*70)
print(f"ANNUAL SAVINGS: €{inv_savings:,.2f}")
print(f"Percentage reduction: {(inv_savings / inv_cost['total_cost'])*100:.1f}%")


AI INVOICE EXTRACTION IMPACT:
AI improvements:
  Touch time reduction: 50% → 6.0 min/invoice
  Rework reduction: 80% → 1.00% rework rate
  Escalation reduction: 80% → 0.40% escalation rate
  AI cost: €0.03 per invoice

WITH AI:
  Manual processing:     € 23,100.00
  Rework:                €    231.00
  Escalations:           €    126.00
  AI platform cost:      €    126.00
  ----------------------------------------
  TOTAL COST:            € 23,583.00/year

ANNUAL SAVINGS: €26,187.00
Percentage reduction: 52.6%


---

## Workflow 2: Support Ticket Triage Automation

**Scenario:** Customer support team manually reviews and routes incoming tickets to correct team (L1 triage, L2 specialist, escalation to product).

### Current State

In [9]:
print("WORKFLOW 2: SUPPORT TICKET TRIAGE")
print("="*70)

# Support triage metrics
sup_volume = 18000  # tickets per year
sup_touch_time = 14  # minutes triage + initial routing
sup_rework_rate = 0.08  # 8% misrouted, need to be retriaged
sup_escalation_rate = 0.15  # 15% complex → specialist
sup_analyst_cost = 55  # €/hr L1 analyst
sup_specialist_cost = 75  # €/hr L2 specialist (does detailed analysis)
sup_escalation_time = 30  # specialist spends 30 min on escalated cases

print(f"\nCURRENT STATE METRICS:")
print(f"  Annual volume: {sup_volume:,} tickets")
print(f"  L1 triage time: {sup_touch_time} minutes per ticket")
print(f"  Rework rate: {sup_rework_rate*100:.0f}% (misrouted tickets)")
print(f"  Escalation rate: {sup_escalation_rate*100:.0f}% (complex → L2 specialist)")
print(f"  L1 Analyst cost: €{sup_analyst_cost}/hr")
print(f"  L2 Specialist cost: €{sup_specialist_cost}/hr")
print(f"  Specialist time per escalated case: {sup_escalation_time} minutes")

# Calculate cost
sup_cost = cost_model_complete(
    volume=sup_volume,
    touch_time_min=sup_touch_time,
    rework_rate=sup_rework_rate,
    escalation_rate=sup_escalation_rate,
    base_hourly_cost=sup_analyst_cost,
    escalation_hourly_cost=sup_specialist_cost,
    escalation_extra_time_min=sup_escalation_time,
)

print(f"\nCOST BREAKDOWN:")
print(f"  L1 analyst triage:       €{sup_cost['base_cost']:>10,.2f}")
print(f"  Rework (retriaging):     €{sup_cost['rework_cost']:>10,.2f}")
print(f"  L2 specialist handling:  €{sup_cost['escalation_cost']:>10,.2f}")
print(f"  " + "-"*40)
print(f"  TOTAL COST OF FRICTION:  €{sup_cost['total_cost']:>10,.2f}/year")
print(f"  Cost per ticket:         €{sup_cost['cost_per_transaction']:>10,.2f}")

WORKFLOW 2: SUPPORT TICKET TRIAGE

CURRENT STATE METRICS:
  Annual volume: 18,000 tickets
  L1 triage time: 14 minutes per ticket
  Rework rate: 8% (misrouted tickets)
  Escalation rate: 15% (complex → L2 specialist)
  L1 Analyst cost: €55/hr
  L2 Specialist cost: €75/hr
  Specialist time per escalated case: 30 minutes

COST BREAKDOWN:
  L1 analyst triage:       €231,000.00
  Rework (retriaging):     € 18,480.00
  L2 specialist handling:  €101,250.00
  ----------------------------------------
  TOTAL COST OF FRICTION:  €350,730.00/year
  Cost per ticket:         €     19.48


In [10]:
# AI Impact
print("\nAI TRIAGE ASSISTANT IMPACT:")
print("="*70)

ai_sup_touch_time_reduction = 0.45  # 45% faster (AI learns routing patterns)
ai_sup_rework_reduction = 0.90  # 90% fewer misroutes (AI learns from feedback)
ai_sup_escalation_reduction = 0.85  # 85% fewer need specialist (AI handles standard cases)
ai_sup_platform_cost = 0.05  # €0.05 per ticket for AI platform

sup_ai_touch_time = sup_touch_time * (1 - ai_sup_touch_time_reduction)
sup_ai_rework_rate = sup_rework_rate * (1 - ai_sup_rework_reduction)
sup_ai_escalation_rate = sup_escalation_rate * (1 - ai_sup_escalation_reduction)

print(f"AI improvements:")
print(f"  Touch time reduction: {ai_sup_touch_time_reduction*100:.0f}% → {sup_ai_touch_time:.1f} min/ticket")
print(f"  Rework reduction: {ai_sup_rework_reduction*100:.0f}% → {sup_ai_rework_rate*100:.2f}% rework rate")
print(f"  Escalation reduction: {ai_sup_escalation_reduction*100:.0f}% → {sup_ai_escalation_rate*100:.2f}% escalation rate")
print(f"  AI cost: €{ai_sup_platform_cost:.2f} per ticket")

# Calculate new cost with AI
sup_cost_ai = cost_model_complete(
    volume=sup_volume,
    touch_time_min=sup_ai_touch_time,
    rework_rate=sup_ai_rework_rate,
    escalation_rate=sup_ai_escalation_rate,
    base_hourly_cost=sup_analyst_cost,
    escalation_hourly_cost=sup_specialist_cost,
    escalation_extra_time_min=sup_escalation_time,
)
ai_sup_platform_total = sup_volume * ai_sup_platform_cost
sup_cost_ai_total = sup_cost_ai['total_cost'] + ai_sup_platform_total

print(f"\nWITH AI:")
print(f"  L1 analyst triage:      €{sup_cost_ai['base_cost']:>10,.2f}")
print(f"  Rework:                 €{sup_cost_ai['rework_cost']:>10,.2f}")
print(f"  L2 specialist:          €{sup_cost_ai['escalation_cost']:>10,.2f}")
print(f"  AI platform cost:       €{ai_sup_platform_total:>10,.2f}")
print(f"  " + "-"*40)
print(f"  TOTAL COST:             €{sup_cost_ai_total:>10,.2f}/year")

sup_savings = sup_cost['total_cost'] - sup_cost_ai_total
print(f"\n" + "="*70)
print(f"ANNUAL SAVINGS: €{sup_savings:,.2f}")
print(f"Percentage reduction: {(sup_savings / sup_cost['total_cost'])*100:.1f}%")


AI TRIAGE ASSISTANT IMPACT:
AI improvements:
  Touch time reduction: 45% → 7.7 min/ticket
  Rework reduction: 90% → 0.80% rework rate
  Escalation reduction: 85% → 2.25% escalation rate
  AI cost: €0.05 per ticket

WITH AI:
  L1 analyst triage:      €127,050.00
  Rework:                 €  1,016.40
  L2 specialist:          € 15,187.50
  AI platform cost:       €    900.00
  ----------------------------------------
  TOTAL COST:             €144,153.90/year

ANNUAL SAVINGS: €206,576.10
Percentage reduction: 58.9%


---

## Workflow 3: Weekly Management Reporting

**Scenario:** Finance manager manually gathers data, analyzes trends, writes weekly executive report to leadership.

### Current State

In [11]:
print("WORKFLOW 3: WEEKLY MANAGEMENT REPORTING")
print("="*70)

# Management reporting metrics
rep_volume = 52  # reports per year (weekly)
rep_touch_time = 180  # minutes per report (data collection + analysis + writing)
rep_rework_rate = 0.30  # 30% require revisions (errors found in executive review)
rep_escalation_rate = 0.05  # 5% need executive director review & revision
finance_manager_cost = 45  # €/hr (€93k/year typical manager)
executive_director_cost = 90  # €/hr (higher-level review)

print(f"\nCURRENT STATE METRICS:")
print(f"  Annual volume: {rep_volume} reports (weekly)")
print(f"  Time per report: {rep_touch_time} minutes (data + analysis + writing)")
print(f"  Rework rate: {rep_rework_rate*100:.0f}% (require revision after review)")
print(f"  Executive escalation rate: {rep_escalation_rate*100:.1f}% (director review)")
print(f"  Finance Manager cost: €{finance_manager_cost}/hr")
print(f"  Executive Director cost: €{executive_director_cost}/hr")

# Calculate cost
rep_cost = cost_model_complete(
    volume=rep_volume,
    touch_time_min=rep_touch_time,
    rework_rate=rep_rework_rate,
    escalation_rate=rep_escalation_rate,
    base_hourly_cost=finance_manager_cost,
    escalation_hourly_cost=executive_director_cost,
)

print(f"\nCOST BREAKDOWN:")
print(f"  Manager preparation:     €{rep_cost['base_cost']:>10,.2f}")
print(f"  Rework/revision:         €{rep_cost['rework_cost']:>10,.2f}")
print(f"  Executive review:        €{rep_cost['escalation_cost']:>10,.2f}")
print(f"  " + "-"*40)
print(f"  TOTAL COST OF FRICTION:  €{rep_cost['total_cost']:>10,.2f}/year")
print(f"  Cost per report:         €{rep_cost['cost_per_transaction']:>10,.2f}")

WORKFLOW 3: WEEKLY MANAGEMENT REPORTING

CURRENT STATE METRICS:
  Annual volume: 52 reports (weekly)
  Time per report: 180 minutes (data + analysis + writing)
  Rework rate: 30% (require revision after review)
  Executive escalation rate: 5.0% (director review)
  Finance Manager cost: €45/hr
  Executive Director cost: €90/hr

COST BREAKDOWN:
  Manager preparation:     €  7,020.00
  Rework/revision:         €  2,106.00
  Executive review:        €    702.00
  ----------------------------------------
  TOTAL COST OF FRICTION:  €  9,828.00/year
  Cost per report:         €    189.00


In [12]:
# AI Impact
print("\nAI REPORTING ASSISTANT IMPACT:")
print("="*70)

ai_rep_touch_time_reduction = 0.60  # 60% faster (AI auto-gathers data from systems)
ai_rep_rework_reduction = 0.90  # 90% fewer errors (AI validates formats, catches issues)
ai_rep_escalation_reduction = 0.70  # 70% fewer executive reviews (AI drafts better-quality reports)
ai_rep_platform_cost = 15  # €15 per report for AI platform

rep_ai_touch_time = rep_touch_time * (1 - ai_rep_touch_time_reduction)
rep_ai_rework_rate = rep_rework_rate * (1 - ai_rep_rework_reduction)
rep_ai_escalation_rate = rep_escalation_rate * (1 - ai_rep_escalation_reduction)

print(f"AI improvements:")
print(f"  Touch time reduction: {ai_rep_touch_time_reduction*100:.0f}% → {rep_ai_touch_time:.0f} min/report")
print(f"  Rework reduction: {ai_rep_rework_reduction*100:.0f}% → {rep_ai_rework_rate*100:.1f}% rework rate")
print(f"  Escalation reduction: {ai_rep_escalation_reduction*100:.0f}% → {rep_ai_escalation_rate*100:.2f}% escalation rate")
print(f"  AI cost: €{ai_rep_platform_cost:.2f} per report")

# Calculate new cost with AI
rep_cost_ai = cost_model_complete(
    volume=rep_volume,
    touch_time_min=rep_ai_touch_time,
    rework_rate=rep_ai_rework_rate,
    escalation_rate=rep_ai_escalation_rate,
    base_hourly_cost=finance_manager_cost,
    escalation_hourly_cost=executive_director_cost,
)
ai_rep_platform_total = rep_volume * ai_rep_platform_cost
rep_cost_ai_total = rep_cost_ai['total_cost'] + ai_rep_platform_total

print(f"\nWITH AI:")
print(f"  Manager preparation:    €{rep_cost_ai['base_cost']:>10,.2f}")
print(f"  Rework/revision:        €{rep_cost_ai['rework_cost']:>10,.2f}")
print(f"  Executive review:       €{rep_cost_ai['escalation_cost']:>10,.2f}")
print(f"  AI platform cost:       €{ai_rep_platform_total:>10,.2f}")
print(f"  " + "-"*40)
print(f"  TOTAL COST:             €{rep_cost_ai_total:>10,.2f}/year")

rep_savings = rep_cost['total_cost'] - rep_cost_ai_total
print(f"\n" + "="*70)
print(f"ANNUAL SAVINGS: €{rep_savings:,.2f}")
print(f"Percentage reduction: {(rep_savings / rep_cost['total_cost'])*100:.1f}%")


AI REPORTING ASSISTANT IMPACT:
AI improvements:
  Touch time reduction: 60% → 72 min/report
  Rework reduction: 90% → 3.0% rework rate
  Escalation reduction: 70% → 1.50% escalation rate
  AI cost: €15.00 per report

WITH AI:
  Manager preparation:    €  2,808.00
  Rework/revision:        €     84.24
  Executive review:       €     84.24
  AI platform cost:       €    780.00
  ----------------------------------------
  TOTAL COST:             €  3,756.48/year

ANNUAL SAVINGS: €6,071.52
Percentage reduction: 61.8%


---

## Part 5: Cost Model Comparison Matrix

How do these three workflows stack up against each other?

In [13]:
# Build comparison matrix
comparison = pd.DataFrame({
    'Workflow': ['Invoice Approval', 'Support Triage', 'Management Reporting'],
    'Annual Volume': [inv_volume, sup_volume, rep_volume],
    'Touch Time (min)': [inv_touch_time, sup_touch_time, rep_touch_time],
    'Baseline Cost (€)': [
        inv_cost['total_cost'],
        sup_cost['total_cost'],
        rep_cost['total_cost']
    ],
    'Cost w/ AI (€)': [
        inv_cost_ai_total,
        sup_cost_ai_total,
        rep_cost_ai_total
    ],
    'Annual Savings (€)': [
        inv_savings,
        sup_savings,
        rep_savings
    ],
    'Savings %': [
        (inv_savings / inv_cost['total_cost'])*100,
        (sup_savings / sup_cost['total_cost'])*100,
        (rep_savings / rep_cost['total_cost'])*100,
    ]
})

print("\nCOMPREHENSIVE COST MODEL COMPARISON")
print("="*120)
print(comparison.to_string(index=False))

total_baseline = comparison['Baseline Cost (€)'].sum()
total_ai = comparison['Cost w/ AI (€)'].sum()
total_savings = comparison['Annual Savings (€)'].sum()

print("\n" + "="*120)
print(f"TOTAL CURRENT COST (3 workflows):    €{total_baseline:>12,.2f}/year")
print(f"TOTAL WITH AI (3 workflows):        €{total_ai:>12,.2f}/year")
print(f"TOTAL SAVINGS OPPORTUNITY:          €{total_savings:>12,.2f}/year ({(total_savings/total_baseline)*100:.1f}%)")
print("="*120)


COMPREHENSIVE COST MODEL COMPARISON
            Workflow  Annual Volume  Touch Time (min)  Baseline Cost (€)  Cost w/ AI (€)  Annual Savings (€)  Savings %
    Invoice Approval           4200                12            49770.0        23583.00            26187.00  52.616034
      Support Triage          18000                14           350730.0       144153.90           206576.10  58.898897
Management Reporting             52               180             9828.0         3756.48             6071.52  61.777778

TOTAL CURRENT COST (3 workflows):    €  410,328.00/year
TOTAL WITH AI (3 workflows):        €  171,493.38/year
TOTAL SAVINGS OPPORTUNITY:          €  238,834.62/year (58.2%)


---

## Part 6: Interactive Exercise

### Your Turn: Build a Cost Model for a New Workflow

Choose ONE of these workflows and calculate the cost model:

In [14]:
print("INTERACTIVE EXERCISE: Pick a Workflow")
print("="*70)
print("\nOption 1: Access Review")
print("  - 2,400 access reviews per year")
print("  - 8 minutes per review")
print("  - 3% escalation to compliance officer (€80/hr)")
print("  - 2% rework rate")

print("\nOption 2: Contract Review (Legal)")
print("  - 180 contracts per year")
print("  - 45 minutes per contract")
print("  - Paralegal cost: €35/hr")
print("  - 25% escalation to attorney (€150/hr)")
print("  - 5% rework rate")

print("\nOption 3: Expense Report Review")
print("  - 3,600 reports per year")
print("  - 6 minutes per report")
print("  - Finance clerk cost: €30/hr")
print("  - 8% escalation to finance manager (€55/hr)")
print("  - 15% rework rate (missing receipts, policy violations)")

print("\n" + "="*70)
print("\nTEMPLATE: Fill in your analysis below")
print("""
Workflow: ___________________________________________

Annual Volume: ________________ 
Touch Time: ________________ minutes
Rework Rate: ________________ %
Escalation Rate: ________________ %
Base Hourly Cost: €____________ /hr
Escalation Hourly Cost: €____________ /hr

Calculations:
- Base cost: €________________
- Rework cost: €________________
- Escalation cost: €________________
- TOTAL: €________________

Answer Key:
  Option 1 (Access Review): €7,920 total (€3.30/review)
  Option 2 (Contract Review): €12,825 total (€71.25/contract)
  Option 3 (Expense Report): €10,152 total (€2.82/report)
""")

INTERACTIVE EXERCISE: Pick a Workflow

Option 1: Access Review
  - 2,400 access reviews per year
  - 8 minutes per review
  - 3% escalation to compliance officer (€80/hr)
  - 2% rework rate

Option 2: Contract Review (Legal)
  - 180 contracts per year
  - 45 minutes per contract
  - Paralegal cost: €35/hr
  - 25% escalation to attorney (€150/hr)
  - 5% rework rate

Option 3: Expense Report Review
  - 3,600 reports per year
  - 6 minutes per report
  - Finance clerk cost: €30/hr
  - 8% escalation to finance manager (€55/hr)
  - 15% rework rate (missing receipts, policy violations)


TEMPLATE: Fill in your analysis below

Workflow: ___________________________________________

Annual Volume: ________________ 
Touch Time: ________________ minutes
Rework Rate: ________________ %
Escalation Rate: ________________ %
Base Hourly Cost: €____________ /hr
Escalation Hourly Cost: €____________ /hr

Calculations:
- Base cost: €________________
- Rework cost: €________________
- Escalation cost: €__

---

## Part 7: Sensitivity Analysis

How sensitive are these cost models to changes in assumptions?

In [15]:
# Sensitivity analysis for Invoice Approval
print("SENSITIVITY ANALYSIS: Invoice Approval")
print("="*70)
print("\nHow does total cost change with different assumptions?")
print("(Base case: 4,200 invoices, 5% rework, €55/hr)\n")

sensitivity = []

# Volume sensitivity
for volume_multiplier in [0.8, 0.9, 1.0, 1.1, 1.2]:
    vol = inv_volume * volume_multiplier
    cost = cost_model_complete(
        volume=vol,
        touch_time_min=inv_touch_time,
        rework_rate=inv_rework_rate,
        escalation_rate=inv_escalation_rate,
        base_hourly_cost=ap_analyst_cost,
        escalation_hourly_cost=ap_manager_cost,
    )
    sensitivity.append({
        'Scenario': f'Volume {volume_multiplier:.0%}',
        'Volume': int(vol),
        'Total Cost': cost['total_cost'],
        'Change from Base': cost['total_cost'] - inv_cost['total_cost']
    })

# Rework rate sensitivity
for rework_mult in [0.5, 0.75, 1.0, 1.5, 2.0]:
    rework = inv_rework_rate * rework_mult
    cost = cost_model_complete(
        volume=inv_volume,
        touch_time_min=inv_touch_time,
        rework_rate=rework,
        escalation_rate=inv_escalation_rate,
        base_hourly_cost=ap_analyst_cost,
        escalation_hourly_cost=ap_manager_cost,
    )
    sensitivity.append({
        'Scenario': f'Rework {rework_mult:.1f}x',
        'Volume': inv_volume,
        'Total Cost': cost['total_cost'],
        'Change from Base': cost['total_cost'] - inv_cost['total_cost']
    })

# Hourly cost sensitivity
for cost_mult in [0.9, 0.95, 1.0, 1.05, 1.1]:
    hourly = ap_analyst_cost * cost_mult
    cost = cost_model_complete(
        volume=inv_volume,
        touch_time_min=inv_touch_time,
        rework_rate=inv_rework_rate,
        escalation_rate=inv_escalation_rate,
        base_hourly_cost=hourly,
        escalation_hourly_cost=ap_manager_cost * cost_mult,
    )
    sensitivity.append({
        'Scenario': f'Hourly Cost {cost_mult:.0%}',
        'Volume': inv_volume,
        'Total Cost': cost['total_cost'],
        'Change from Base': cost['total_cost'] - inv_cost['total_cost']
    })

sens_df = pd.DataFrame(sensitivity)
print(sens_df.to_string(index=False))

print("\n" + "="*70)
print("KEY INSIGHT:")
print(f"  • ±20% volume change → ±€{inv_cost['base_cost']*0.2/12*20:,.0f} cost change")
print(f"  • 2x rework rate → +€{(inv_rework_rate * inv_touch_time/60 * ap_analyst_cost * inv_volume):,.0f} additional cost")
print(f"  • +10% hourly cost → +€{inv_cost['total_cost']*0.1:,.0f} additional cost")
print("\nSmall changes in volume and rework rate create BIG financial differences!")

SENSITIVITY ANALYSIS: Invoice Approval

How does total cost change with different assumptions?
(Base case: 4,200 invoices, 5% rework, €55/hr)

        Scenario  Volume  Total Cost  Change from Base
      Volume 80%    3360     39816.0           -9954.0
      Volume 90%    3780     44793.0           -4977.0
     Volume 100%    4200     49770.0               0.0
     Volume 110%    4620     54747.0            4977.0
     Volume 120%    5040     59724.0            9954.0
     Rework 0.5x    4200     48615.0           -1155.0
     Rework 0.8x    4200     49192.5            -577.5
     Rework 1.0x    4200     49770.0               0.0
     Rework 1.5x    4200     50925.0            1155.0
     Rework 2.0x    4200     52080.0            2310.0
 Hourly Cost 90%    4200     44793.0           -4977.0
 Hourly Cost 95%    4200     47281.5           -2488.5
Hourly Cost 100%    4200     49770.0               0.0
Hourly Cost 105%    4200     52258.5            2488.5
Hourly Cost 110%    4200     547

---

## ✓ Lesson 02 Complete

You can now:
- ✓ Build cost models that account for base cost + rework + escalation
- ✓ Calculate cost per transaction and total annual cost of friction
- ✓ Compare workflows by financial opportunity
- ✓ Model AI impact on workflow costs
- ✓ Run sensitivity analysis to understand cost drivers

**Next:** Lesson 03 — EBITDA Basics and the EBITDA Bridge

Ready? Let's go! 🚀